<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_LotkaVolterra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 9b — Lotka–Volterra

This notebook runs only the `lotka_volterra` SBIBM task. It delegates the ML experiment to the shared Exercise-9b runner, so it retains exactly the same flow topology, four-member flow ensemble, ten-member plain classifier ensemble, pure-CE objective, 250-epoch stepped learning-rate campaign, checkpoint validation, metrics, and artifact schema as the other task notebooks.

## Important: start from a fresh Colab runtime

If you previously attempted the Julia installation, select **Runtime → Restart session**, reopen this notebook if necessary, and then choose **Runtime → Run all**. No mid-notebook restart and no Julia installation are required. The source checkout is runtime-local; models, simulation banks, results, and standalone figure scripts are saved under `MyDrive/hybrid_nsbi_ml/exercise_9b_SBIBM`.

## Why this notebook does not use `diffeqtorch`

`sbibm==1.1.0` implements Lotka–Volterra through the now-legacy `diffeqtorch`/PyJulia/SciML stack, which is not compatible with the current Colab runtime. Here we replace **only the numerical ODE solver**. The statistical task is unchanged:

$$\dot x=\alpha x-\beta xy,\qquad \dot y=-\gamma y+\delta xy,$$

with the official initial state $(x_0,y_0)=(30,1)$, the official four-dimensional LogNormal prior, summaries at days $0,2.1,\ldots,18.9$, and the official LogNormal observation model with log-scale width 0.1. Official observations and official reference-posterior samples are still read directly from `sbibm`.

The replacement integrates the same equations in log-population coordinates, $u=\log x$ and $v=\log y$, for which

$$\dot u=\alpha-\beta e^v,\qquad \dot v=-\gamma+\delta e^u.$$

This exact change of variables preserves positive populations and is numerically stable across the broad prior. We use vectorized fourth-order Runge–Kutta integration with $\Delta t=0.0025$ day. Before any simulations or training, the notebook compares it with high-accuracy DOP853 solutions at the prior center and all 16 crossed four-standard-deviation corners. The run stops unless the maximum error in the LogNormal location is below $5\times10^{-5}$. The measured audit error is approximately $2.9\times10^{-5}$, only $2.9\times10^{-4}$ of the observation-noise standard deviation. The backend identifier and audit are written into the task status JSON and result artifacts.

After this finishes, run `Exercise_9b_SBIBM.ipynb` to include Lotka–Volterra in the partial or complete comparison.

In [ ]:
TASK_NAME = "lotka_volterra"
PROFILE = "PAPER"       # SMOKE | TUTORIAL | PAPER
BASE_SEED = 29082026
LOAD_IF_AVAILABLE = False
FAIL_ON_TASK_ERROR = True

import os
os.environ["EX9B_TASK"] = TASK_NAME
os.environ["EX9B_PROFILE"] = PROFILE
os.environ["EX9B_SEED"] = str(BASE_SEED)
os.environ["EX9B_LOAD_IF_AVAILABLE"] = "1" if LOAD_IF_AVAILABLE else "0"
os.environ["EX9B_FAIL_ON_TASK_ERROR"] = "1" if FAIL_ON_TASK_ERROR else "0"
os.environ["EX9B_LOTKA_VOLTERRA_BACKEND"] = "python_logrk4"

## Run the audited Lotka–Volterra task

The shared engine installs the lightweight Python dependencies, activates and audits the compatibility backend, generates or loads the exact 10,000-pair training bank, and then starts the matched JANA-versus-hybrid experiment. A line beginning with `Simulator backend preflight:` must appear before training.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"

if "google.colab" in sys.modules:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none",
            "--sparse", "--branch", BRANCH, REPO_URL, str(repository),
        ], check=True, env=clone_env)
    else:
        subprocess.run([
            "git", "-C", str(repository), "fetch", "origin", BRANCH,
        ], check=True)
        subprocess.run([
            "git", "-C", str(repository), "checkout", BRANCH,
        ], check=True)
        subprocess.run([
            "git", "-C", str(repository), "pull", "--ff-only",
            "origin", BRANCH,
        ], check=True)
    subprocess.run([
        "git", "-C", str(repository), "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    ], check=True)
    runner = (
        repository / "workshops" / "ml4hep_tifr_colab"
        / "Exercise_9b_SBIBM_Core.ipynb"
    )
else:
    candidates = [
        Path.cwd() / "Exercise_9b_SBIBM_Core.ipynb",
        Path.cwd() / "workshops" / "ml4hep_tifr_colab"
        / "Exercise_9b_SBIBM_Core.ipynb",
    ]
    runner = next((path for path in candidates if path.exists()), None)
    if runner is None:
        raise FileNotFoundError("Cannot locate Exercise_9b_SBIBM_Core.ipynb")

print("Running shared task engine:", runner)
get_ipython().run_line_magic("run", f'"{runner}"')